# QSAR Training Pipeline — All Targets

This notebook trains QSAR models for four therapeutic targets:

| Target | ChEMBL ID | Disease Area |
|--------|-----------|--------------|
| Thrombin | CHEMBL204 | Blood coagulation |
| BACE-1 | CHEMBL4822 | Alzheimer's disease |
| CDK2 | CHEMBL301 | Cancer (cell cycle) |
| EGFR | CHEMBL203 | Cancer (growth factor) |

**How to use:**
1. Set `ACTIVE_TARGET` in the Configuration cell
2. Run All Cells (Kernel → Restart & Run All)
3. A `.pkl` file is saved automatically in the `models/` folder
4. Repeat for each target

> All models use the same pipeline: Morgan Fingerprints (r=2, 2048 bits) + XGBoost with scale_pos_weight, with automatic selection of the best model by ROC-AUC.


## 0. Install Dependencies

In [1]:
import sys
# Uncomment and run once if any package is missing:
# !{sys.executable} -m pip install pandas numpy joblib scikit-learn chembl-webresource-client matplotlib rdkit xgboost


## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import time
import requests
from pathlib import Path
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.metrics import (classification_report, roc_auc_score,
                              ConfusionMatrixDisplay)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("All imports OK.")


All imports OK.


## 2. Configuration

**Change only this cell** to train a different target.  
Set `ACTIVE_TARGET` to one of: `"thrombin"`, `"bace1"`, `"cdk2"`, `"egfr"`


In [3]:
TARGETS = {
    "thrombin": {
        "chembl_id":         "CHEMBL204",
        "ic50_threshold_um": 10.0,
        "model_path":        "models/model_thrombin.pkl",
        "description":       "Thrombin — Serine protease involved in blood coagulation",
    },
    "bace1": {
        "chembl_id":         "CHEMBL4822",
        "ic50_threshold_um": 10.0,
        "model_path":        "models/model_bace1.pkl",
        "description":       "BACE-1 — Beta-secretase 1, protease involved in Alzheimer's disease",
    },
    "cdk2": {
        "chembl_id":         "CHEMBL301",
        "ic50_threshold_um": 10.0,
        "model_path":        "models/model_cdk2.pkl",
        "description":       "CDK2 — Cyclin-dependent kinase 2, involved in cell cycle and cancer",
    },
    "egfr": {
        "chembl_id":         "CHEMBL203",
        "ic50_threshold_um": 10.0,
        "model_path":        "models/model_egfr.pkl",
        "description":       "EGFR — Epidermal growth factor receptor, cancer target",
    },
}

# ── Change this to "thrombin", "bace1", "cdk2", or "egfr" ───────────────────
ACTIVE_TARGET = "bace1"

cfg = TARGETS[ACTIVE_TARGET]

FP_RADIUS = 2
FP_NBITS  = 2048

# Create models/ folder if it doesn't exist
Path("models").mkdir(exist_ok=True)

print(f"Active target : {ACTIVE_TARGET}")
print(f"ChEMBL ID     : {cfg['chembl_id']}")
print(f"Description   : {cfg['description']}")
print(f"IC50 threshold: {cfg['ic50_threshold_um']} µM")
print(f"Model output  : {cfg['model_path']}")
print(f"Fingerprint   : Morgan r={FP_RADIUS}, {FP_NBITS} bits")


Active target : bace1
ChEMBL ID     : CHEMBL4822
Description   : BACE-1 — Beta-secretase 1, protease involved in Alzheimer's disease
IC50 threshold: 10.0 µM
Model output  : models/model_bace1.pkl
Fingerprint   : Morgan r=2, 2048 bits


## 3. Download Bioactivity Data from ChEMBL

Fetches all IC₅₀ records for the active target with **automatic retries** and
**local CSV cache**. If a previous run was interrupted, the cached file is
loaded instantly — no re-download needed.

**Cache file:** `data/<chembl_id>_ic50_raw.csv`  
Delete it manually if you want to force a fresh download.


In [4]:
# ── Download settings ────────────────────────────────────────────────────────
MAX_RETRIES   = 5          # Number of retry attempts per page
BACKOFF_BASE  = 2          # Exponential backoff base (seconds)
PAGE_SIZE     = 1000       # Records per page (ChEMBL default max)
FIELDS        = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "assay_chembl_id",
]

Path("data").mkdir(exist_ok=True)
cache_path = Path(f"data/{cfg['chembl_id']}_ic50_raw.csv")


def fetch_page_with_retry(queryset, offset, limit, max_retries=MAX_RETRIES):
    """Fetch a single page from ChEMBL with exponential backoff on failure."""
    for attempt in range(1, max_retries + 1):
        try:
            page = list(queryset[offset : offset + limit])
            return page
        except Exception as e:
            wait = BACKOFF_BASE ** attempt
            print(
                f"  ⚠️  Attempt {attempt}/{max_retries} failed at offset {offset}: "
                f"{type(e).__name__}: {e}"
            )
            if attempt == max_retries:
                raise RuntimeError(
                    f"Download failed after {max_retries} attempts at offset {offset}. "
                    "Try again later or reduce PAGE_SIZE."
                ) from e
            print(f"  ⏳ Waiting {wait}s before retry...")
            time.sleep(wait)
    return []  # unreachable, satisfies linters


def download_chembl_data(chembl_id: str, cache_path: Path) -> pd.DataFrame:
    """Download IC50 data with pagination, retries, and local CSV cache."""

    # ── 1. Load from cache if it exists ──────────────────────────────────────
    if cache_path.exists():
        print(f"✅ Cache found: {cache_path}")
        print("   Loading from cache (delete file to force re-download)...")
        df_cached = pd.read_csv(cache_path)
        print(f"   Records loaded: {len(df_cached):,}")
        return df_cached

    # ── 2. Build queryset ─────────────────────────────────────────────────────
    print(f"Fetching IC50 data for {chembl_id} from ChEMBL...")
    activity = new_client.activity
    queryset = activity.filter(
        target_chembl_id=chembl_id,
        standard_type="IC50",
        relation="=",
    ).only(FIELDS)

    # ── 3. Get total count ────────────────────────────────────────────────────
    print("   Getting total record count...")
    try:
        total = len(queryset)
    except Exception as e:
        raise RuntimeError(f"Could not get record count from ChEMBL: {e}") from e
    print(f"   Total records available: {total:,}")

    # ── 4. Paginated download with retries ────────────────────────────────────
    all_records = []
    offset      = 0
    page_num    = 0

    while offset < total:
        page_num += 1
        print(f"   Page {page_num:>3} | offset {offset:>6}/{total:,} ", end="", flush=True)

        page = fetch_page_with_retry(queryset, offset, PAGE_SIZE)

        all_records.extend(page)
        offset += PAGE_SIZE
        print(f"| +{len(page)} records (total so far: {len(all_records):,})")

        # Polite pause between pages to avoid rate-limit cutoffs
        if offset < total:
            time.sleep(0.5)

    # ── 5. Build DataFrame and save cache ────────────────────────────────────
    df = pd.DataFrame.from_records(all_records)
    df.to_csv(cache_path, index=False)
    print(f"\n✅ Download complete: {len(df):,} records")
    print(f"   Cache saved → {cache_path}")
    return df


# ── Run the download ──────────────────────────────────────────────────────────
df_raw = download_chembl_data(cfg["chembl_id"], cache_path)

print(f"\nColumns: {list(df_raw.columns)}")
df_raw.head(3)


Fetching IC50 data for CHEMBL4822 (bace1) from ChEMBL...


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

## 4. Clean the Dataset

- Drop missing SMILES / IC₅₀
- Keep only nM and µM units, convert all to µM
- Remove duplicates (median IC₅₀ per molecule)
- Label: **Active** (IC₅₀ < threshold) / **Inactive**


In [ ]:
df = df_raw.copy()

df = df.dropna(subset=["canonical_smiles", "standard_value"])
df = df[df["standard_units"].isin(["nM", "uM", "µM"])]
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])

df["ic50_um"] = df.apply(
    lambda row: row["standard_value"] / 1000
    if row["standard_units"] == "nM"
    else row["standard_value"],
    axis=1
)

df = (
    df.groupby("molecule_chembl_id", as_index=False)
    .agg(
        canonical_smiles=("canonical_smiles", "first"),
        ic50_um=("ic50_um", "median"),
    )
)

threshold = cfg["ic50_threshold_um"]
df["label"] = (df["ic50_um"] < threshold).astype(int)

print(f"Clean dataset shape : {df.shape}")
print(f"Active   (label=1)  : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"Inactive (label=0)  : {(df['label']==0).sum()} ({(df['label']==0).mean()*100:.1f}%)")
df.head(3)


## 5. Generate Morgan Fingerprints

Uses the new `MorganGenerator` API (no deprecation warnings).


In [ ]:
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)

def smiles_to_morgan(smiles):
    """Convert SMILES to Morgan fingerprint array. Returns None if invalid."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return np.array(morgan_gen.GetFingerprint(mol))

fps = df["canonical_smiles"].apply(smiles_to_morgan)

valid_mask = fps.notna()
df  = df[valid_mask].reset_index(drop=True)
fps = fps[valid_mask].reset_index(drop=True)

X = np.vstack(fps.values)
y = df["label"].values

print(f"Feature matrix shape    : {X.shape}")
print(f"Label distribution      : {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Dropped (invalid SMILES): {(~valid_mask).sum()}")


## 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test : {X_test.shape[0]} samples")


## 7. Baseline Model — Gradient Boosting

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
)

print("Training Baseline GradientBoosting...")
model.fit(X_train, y_train)
print("Training complete.")


## 8. Baseline Evaluation

In [ ]:
y_pred       = model.predict(X_test)
y_proba      = model.predict_proba(X_test)[:, 1]
roc_baseline = roc_auc_score(y_test, y_proba)

print("=== Baseline GradientBoosting — Test Set ===")
print(classification_report(y_test, y_pred, target_names=["Inactive", "Active"]))
print(f"ROC-AUC: {roc_baseline:.4f}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
print(f"5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Inactive", "Active"],
    ax=ax, colorbar=False
)
ax.set_title(f"Baseline GradientBoosting — {ACTIVE_TARGET.upper()}")
plt.tight_layout()
plt.show()


## 9. Tuned Random Forest — class_weight='balanced'

Corrects class imbalance and optimises hyperparameters with RandomizedSearchCV.


In [ ]:
param_dist_rf = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [3, 4, 5, 6, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2", 0.3, 0.5],
}

rf_balanced = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_search = RandomizedSearchCV(
    estimator=rf_balanced,
    param_distributions=param_dist_rf,
    n_iter=50,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("Running RandomizedSearchCV for Random Forest...")
rf_search.fit(X_train, y_train)

print(f"\nBest parameters : {rf_search.best_params_}")
print(f"Best CV ROC-AUC : {rf_search.best_score_:.4f}")

tuned_rf      = rf_search.best_estimator_
y_pred_tuned  = tuned_rf.predict(X_test)
y_proba_tuned = tuned_rf.predict_proba(X_test)[:, 1]
roc_rf        = roc_auc_score(y_test, y_proba_tuned)

print(f"\n=== Tuned RandomForest (balanced) — Test Set ===")
print(classification_report(y_test, y_pred_tuned, target_names=["Inactive", "Active"]))
print(f"ROC-AUC : {roc_rf:.4f}")
print(f"Baseline: {roc_baseline:.4f}  →  Delta: {roc_rf - roc_baseline:+.4f}")


## 10. XGBoost — scale_pos_weight + RandomizedSearchCV

`scale_pos_weight` corrects class imbalance natively.


In [ ]:
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale:.4f}  ({(y_train==0).sum()} Inactive / {(y_train==1).sum()} Active)")

param_dist_xgb = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma":            [0, 0.1, 0.3],
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=50,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("Running XGBoost RandomizedSearchCV (this may take a few minutes)...")
xgb_search.fit(X_train, y_train)

print(f"\nBest parameters : {xgb_search.best_params_}")
print(f"Best CV ROC-AUC : {xgb_search.best_score_:.4f}")

xgb_tuned   = xgb_search.best_estimator_
y_pred_xgb  = xgb_tuned.predict(X_test)
y_proba_xgb = xgb_tuned.predict_proba(X_test)[:, 1]
roc_xgb     = roc_auc_score(y_test, y_proba_xgb)

print(f"\n=== XGBoost (scale_pos_weight) — Test Set ===")
print(classification_report(y_test, y_pred_xgb, target_names=["Inactive", "Active"]))
print(f"ROC-AUC  : {roc_xgb:.4f}")
print(f"Baseline : {roc_baseline:.4f}  →  Delta vs Baseline : {roc_xgb - roc_baseline:+.4f}")
print(f"Tuned RF : {roc_rf:.4f}  →  Delta vs Tuned RF : {roc_xgb - roc_rf:+.4f}")


## 11. Model Comparison — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Inactive", "Active"],
    ax=axes[0], colorbar=False
)
axes[0].set_title("Baseline GradientBoosting")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tuned,
    display_labels=["Inactive", "Active"],
    ax=axes[1], colorbar=False
)
axes[1].set_title("Tuned RF (balanced)")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb,
    display_labels=["Inactive", "Active"],
    ax=axes[2], colorbar=False
)
axes[2].set_title("XGBoost (scale_pos_weight)")

plt.suptitle(f"Target: {ACTIVE_TARGET.upper()} ({cfg['chembl_id']})", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n{'Model':<30} {'ROC-AUC':>10}")
print("-" * 42)
print(f"{'Baseline GradientBoosting':<30} {roc_baseline:>10.4f}")
print(f"{'Tuned RF (balanced)':<30} {roc_rf:>10.4f}")
print(f"{'XGBoost (scale_pos_weight)':<30} {roc_xgb:>10.4f}")


## 12. Save Best Model

Automatically selects the model with the highest ROC-AUC and saves it to `models/`.


In [ ]:
candidates = {
    "GradientBoosting":         (model,     roc_baseline),
    "TunedRF_balanced":         (tuned_rf,  roc_rf),
    "XGBoost_scale_pos_weight": (xgb_tuned, roc_xgb),
}

best_name  = max(candidates, key=lambda k: candidates[k][1])
best_model = candidates[best_name][0]
best_roc   = candidates[best_name][1]

print(f"Best model   : {best_name}")
print(f"Best ROC-AUC : {best_roc:.4f}")

output_path = cfg["model_path"]
joblib.dump(best_model, output_path)
print(f"Model saved  : {output_path}")

# Verify
loaded    = joblib.load(output_path)
test_pred = loaded.predict(X_test[:3])
print(f"Load test    : predictions on 3 samples → {test_pred}")
print(f"\n✅ {ACTIVE_TARGET.upper()} model ready for Streamlit integration.")
print(f"   Next step: change ACTIVE_TARGET and re-run for the next target.")


## Next Steps

**Train all targets in order:**

| Step | `ACTIVE_TARGET` | Output file |
|------|----------------|-------------|
| 1 | `"thrombin"` | `models/model_thrombin.pkl` |
| 2 | `"bace1"` | `models/model_bace1.pkl` |
| 3 | `"cdk2"` | `models/model_cdk2.pkl` |
| 4 | `"egfr"` | `models/model_egfr.pkl` |

For each: change `ACTIVE_TARGET` → **Kernel → Restart & Run All** → wait for the ✅ message.

**After all four models are trained**, copy the `models/` folder next to `app.py`.  
The Streamlit app will automatically load all available models in the target selector.
